Logistic regression predicts a probability (0 to 1) for classification — e.g. "will this passenger survive?" We can't use raw wx+b directly since it can be any number; we squash it through sigmoid.

Sigmoid function
σ(z) = 1 / (1 + e⁻ᶻ)      where z = w·x + b
Sigmoid takes any real number and squashes it to between 0 and 1 — perfect for representing a probability. At z=0, σ=0.5. As z→∞, σ→1. As z→−∞, σ→0.

Loss function — Binary Cross-Entropy (not MSE!)
L = −(1/n) Σ [yᵢ·log(ŷᵢ) + (1−yᵢ)·log(1−ŷᵢ)]
Interview question — very common: "Why not use MSE for logistic regression?" Answer: MSE with sigmoid creates a non-convex loss surface — gradient descent can get stuck in local minima. Cross-entropy is convex for logistic regression, guaranteeing convergence to the global minimum, and it penalises confident wrong predictions much more heavily.

Gradient — the elegant part
∂L/∂w = (1/n) Xᵗ(ŷ − y)
This looks IDENTICAL to linear regression's gradient. That's not a coincidence — it's a beautiful property of using cross-entropy loss with sigmoid. If asked in an interview "derive the gradient of logistic regression," this is the answer they want to see, and the fact that it simplifies this cleanly is worth mentioning

In [15]:
class LogisticRegressionScratch:
    def __init__(self, lr=0.01, n_iters=1000):
        self.lr = lr
        self.n_iters = n_iters
        self.w = None
        self.b = 0

    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))

    def fit(self, X, y):
        n_samples, n_features = X.shape
        self.w = np.zeros(n_features)
        self.b = 0

        for _ in range(self.n_iters):
            # Forward pass: linear then sigmoid
            z = np.dot(X, self.w) + self.b
            y_pred = self.sigmoid(z)

            # Gradients — same shape as linear regression!
            dw = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)

            self.w -= self.lr * dw
            self.b -= self.lr * db

    def predict_proba(self, X):
        z = np.dot(X, self.w) + self.b
        return self.sigmoid(z)

    def predict(self, X, threshold=0.5):
        return (self.predict_proba(X) >= threshold).astype(int)

In [17]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import StandardScaler

url = 'https://raw.githubusercontent.com/datasciencedojo/datasets/master/titanic.csv'
df = pd.read_csv(url)

# Minimal preprocessing — only what's needed for the math to work
df['Age'] = df['Age'].fillna(df['Age'].median())
df['Fare'] = df['Fare'].fillna(df['Fare'].median())

print(df.isnull().sum())  # Age and Fare should now show 0

PassengerId      0
Survived         0
Pclass           0
Name             0
Sex              0
Age              0
SibSp            0
Parch            0
Ticket           0
Fare             0
Cabin          687
Embarked         2
dtype: int64


In [18]:
# Test on Titanic — predict Survived from Age, Pclass, Fare
from sklearn.linear_model import LogisticRegression as SklearnLogReg
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

X = df[['Age', 'Pclass', 'Fare']].values
y = df['Survived'].values

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=42)

# Your scratch model
my_model = LogisticRegressionScratch(lr=0.1, n_iters=2000)
my_model.fit(X_train, y_train)
my_preds = my_model.predict(X_test)
print("My accuracy:", accuracy_score(y_test, my_preds))

# Sklearn's model
sk_model = SklearnLogReg()
sk_model.fit(X_train, y_train)
sk_preds = sk_model.predict(X_test)
print("Sklearn accuracy:", accuracy_score(y_test, sk_preds))

My accuracy: 0.7374301675977654
Sklearn accuracy: 0.7374301675977654
